# Day 38: Trees & SVM: Phase 3 Assessment
**Author:** Sahil-K-Y  
**Phase:** 3 - Tree Models & SVM  
**Date:** Day 038

---

## Assessment Guidelines

Welcome to the **Phase 3 Assessment**. This notebook serves to evaluate your theoretical understanding and practical implementation skills regarding:
1. Decision Trees (Classification and Regression)
2. Support Vector Machines (SVM) & Kernel Trick
3. Naive Bayes Classifier
4. K-Nearest Neighbors (KNN)
5. Ensemble Methods (Voting, Bagging, Random Forest)
6. Feature Importance and Selection

### Rules & Criteria:
* Your code must execute successfully without throwing errors.
* Provide written explanations/answers where requested. Write your answers as detailed comments inside the designated cells.
* Ensure all plots are fully labeled (titles, axis labels).
* **Total Marks: 25 (5 marks per question)**


## Question 1: Model Selection Scenarios (5 Marks)

Suppose you are tasked with designing machine learning solutions for a banking institution. Look at the scenarios below and answer the questions.

### Scenarios:
* **Scenario A:** Predict credit card fraud detection on a dataset of 1,000,000 transactions with 30 features.
* **Scenario B:** Classify cell types from genomic sequences on a dataset of 5,000 instances with 20,000 features.

### Questions to answer:
1. For Scenario A, which algorithm would you avoid (SVM, Decision Tree, or Random Forest) and why?
2. For Scenario B, which algorithm is suited to handle the high dimensionality, and what preprocessing is mandatory before training SVM/KNN on it?
3. Explain why a single Decision Tree typically overfits, and detail how Random Forest mitigates this issue.
4. What is the fundamental difference in output between Hard Voting and Soft Voting? Under what conditions will Soft Voting out-perform Hard Voting?

**Write your answers as detailed multi-line comments in the code cell below.**


In [ ]:
# Q1 Answers:
# 1. Scenario A: I would avoid SVM.
#    Reason: SVM training complexity is roughly O(N^2) to O(N^3) with respect to the number
#    of samples. For N = 1,000,000, SVM training would be extremely slow, taking hours or
#    days, and would require enormous memory. Decision Tree or Random Forest are much better
#    suited for large datasets.
#
# 2. Scenario B: SVM with a linear kernel or Naive Bayes would be well-suited.
#    Reason: High-dimensional space (20,000 features > 5,000 samples) often allows data to be
#    linearly separable. Support Vector Machine handles high-dimensional sparse/dense data
#    exceedingly well.
#    Mandatory Preprocessing: Feature scaling (e.g., StandardScaler) is mandatory before
#    training SVM or KNN, as they are distance-based models. Without scaling, features with
#    larger scales will dominate the objective function and distance computations.
#
# 3. Overfitting in Decision Trees vs. Random Forests:
#    - Decision Trees split the data until all nodes are pure, resulting in deep, complex
#      trees that capture noise and outliers (high variance).
#    - Random Forest mitigates this by combining multiple independent trees. Each tree is
#      trained on a bootstrap sample of the data (bagging) and splits nodes using only a
#      random subset of features (feature subsampling). Averaging the predictions reduces
#      variance without increasing bias.
#
# 4. Hard vs. Soft Voting:
#    - Hard Voting takes a simple majority vote of the predicted class labels.
#    - Soft Voting calculates the average predicted probability for each class and selects the
#      class with the highest average.
#    - Soft Voting performs better when some classifiers are highly confident and others are
#      uncertain. It allows confident predictions to override uncertain ones.
print("Question 1 completed.")


## Question 2: Compare 4 Classifiers on the Wine Dataset (5 Marks)

Write a clean comparison pipeline to train and evaluate:
1. Decision Tree (max_depth=4)
2. SVM (C=1.0, kernel='rbf')
3. KNN (n_neighbors=5)
4. Random Forest (n_estimators=100)

### Step 2.1: Load Dataset and Split
Let's load the data and create train-test partitions.


In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

wine = load_wine()
X_wine, y_wine = wine.data, wine.target
feat_w, class_w = wine.feature_names, wine.target_names

X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=42, stratify=y_wine
)
print("Data loaded and partitioned.")


### Step 2.2: Scaling Preprocessing
Scale variables using StandardScaler. Keep train and test scaled sets separated.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_w)
X_te_scaled = scaler.transform(X_te_w)
print("Standardization completed.")


### Step 2.3: Initialize Models
We define base estimators in a dictionary, specifying if scaling is needed.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

models_q2 = {
    'Decision Tree': (DecisionTreeClassifier(max_depth=4, random_state=42), False),
    'SVM (RBF)': (SVC(C=1.0, kernel='rbf', random_state=42), True),
    'KNN': (KNeighborsClassifier(n_neighbors=5), True),
    'Random Forest': (RandomForestClassifier(n_estimators=100, random_state=42), False)
}
print("Models defined.")


### Step 2.4: Evaluate and Predict Loop
Let's run 5-fold cross validation and evaluate accuracy scores.


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
import pandas as pd

results_q2 = []

for name, (model, requires_scaling) in models_q2.items():
    X_train_model = X_tr_scaled if requires_scaling else X_tr_w
    X_test_model = X_te_scaled if requires_scaling else X_te_w
    
    cv_scores = cross_val_score(model, X_train_model, y_tr_w, cv=5, scoring='accuracy')
    cv_mean = cv_scores.mean()
    
    model.fit(X_train_model, y_tr_w)
    y_pred = model.predict(X_test_model)
    test_acc = accuracy_score(y_te_w, y_pred)
    
    results_q2.append({
        'Model': name,
        'CV Accuracy (Mean)': cv_mean,
        'Test Accuracy': test_acc
    })

df_results_q2 = pd.DataFrame(results_q2).sort_values(by='Test Accuracy', ascending=False)
print(df_results_q2.to_string(index=False))


## Question 3: SVM vs. Random Forest Hyperparameter Tuning (5 Marks)

Using the **Breast Cancer Dataset**, tune SVM and Random Forest using `GridSearchCV`.

### Grids:
* **SVM:** `C`: [0.1, 1, 10], `gamma`: ['scale', 0.01, 0.1], `kernel`: ['rbf']
* **Random Forest:** `n_estimators`: [50, 100, 200], `max_depth`: [3, 5, None]

### Step 3.1: Load dataset and scale
Let's split the breast cancer dataset.


In [ ]:
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_bc, y_bc, test_size=0.3, random_state=42, stratify=y_bc
)

scaler_bc = StandardScaler()
X_tr_b_scaled = scaler_bc.fit_transform(X_tr_b)
X_te_b_scaled = scaler_bc.transform(X_te_b)
print("Data scaled and prepared.")


### Step 3.2: GridSearch Tuning SVM
Let's run cross validation over the C and gamma parameter space for SVM.


In [ ]:
from sklearn.model_selection import GridSearchCV

svm_param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.1],
    'kernel': ['rbf']
}
svm_grid = GridSearchCV(SVC(random_state=42), svm_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
svm_grid.fit(X_tr_b_scaled, y_tr_b)

best_svm = svm_grid.best_estimator_
svm_test_acc = accuracy_score(y_te_b, best_svm.predict(X_te_b_scaled))
print(f"SVM Tuned CV Score: {svm_grid.best_score_:.4%}")


### Step 3.3: GridSearch Tuning Random Forest
Let's run grid search over trees and maximum depth constraints.


In [ ]:
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, None]
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_tr_b, y_tr_b)

best_rf = rf_grid.best_estimator_
rf_test_acc = accuracy_score(y_te_b, best_rf.predict(X_te_b))
print(f"RF Tuned CV Score: {rf_grid.best_score_:.4%}")


### Step 3.4: Compare Performance
Let's print parameters and check the winner.


In [ ]:
print("===== Tuning Results =====")
print(f"SVM Best Params: {svm_grid.best_params_} | Test Accuracy: {svm_test_acc:.4%}")
print(f"RF Best Params:  {rf_grid.best_params_} | Test Accuracy: {rf_test_acc:.4%}")

winner = "Random Forest" if rf_test_acc > svm_test_acc else "SVM"
print(f"Winner on Test Set: {winner}")


## Question 4: Feature Importance Analysis (5 Marks)

Train a Random Forest Classifier on the Wine dataset, extract the top 5 most important features, plot them, and explain the domain science behind why these features matter for wine classification.

### Step 4.1: Train Model and Extract Importances
Fit model and locate top 5 features.


In [ ]:
rf_wine = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
rf_wine.fit(X_tr_w, y_tr_w)

importances = rf_wine.feature_importances_
top_5_idx = np.argsort(importances)[::-1][:5]
top_5_features = [feat_w[i] for i in top_5_idx]
top_5_importances = importances[top_5_idx]

for rank, (feat, score) in enumerate(zip(top_5_features, top_5_importances), 1):
    print(f"Rank {rank}: {feat:<20} | Score: {score:.4%}")


### Step 4.2: Plot Importances
Plot top 5 features.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.barplot(x=top_5_importances, y=top_5_features, palette='coolwarm')
plt.title('Top 5 Feature Importances (Wine Dataset)', fontsize=14, fontweight='bold')
plt.xlabel('Gini Importance Score')
plt.show()


### Step 4.3: Written Explanation
Provide written explanation of domain science.


In [ ]:
# Q4 Written Explanation:
# Explain the domain science behind the top features (e.g., Flavanoids, Proline, Alcohol, Hue):
# 1. Flavanoids: Flavanoids are natural antioxidants and phenolic compounds found in grape skins.
#    Since different wine cultivars (e.g. Barolo vs. Grignolino) use grapes with different skin
#    properties and have different maceration times, the flavanoid content is highly distinctive.
#
# 2. Proline: Proline is an amino acid present in grapes. The nitrogen profile and amino acid
#    composition (especially proline concentration) are unique biological markers for different
#    grape varieties and soil characteristics, allowing clean separation of cultivars.
#
# 3. Alcohol: The sugar concentration in grapes determines the alcohol percentage after
#    fermentation. Different cultivars are harvested at different ripeness levels, leading to
#    systematic variations in alcohol content.
#
# 4. Color Intensity: Directly relates to the pigmentation (anthocyanin concentration) in grapes.
#    Red cultivars will have vastly different intensity curves than white or lighter red ones.
#
# 5. Hue: Measures the shade/color of the wine. It varies with wine pH, aging potential, and the
#    specific grape skin phenols, serving as a reliable physical marker.
print("Question 4 explanation submitted.")


## Question 5: Open-Ended Classification Challenge (5 Marks)

Build the absolute best performing model you can on the Wine dataset. You are permitted to use:
* Feature scaling and preprocessing
* Custom feature selection
* Hyperparameter tuning (Grid/Randomized Search)
* Ensemble configurations (Voting, Bagging, or Random Forest)

**Goal: Reach or exceed 98% Test Set Accuracy.**

### Step 5.1: Create Optimized Classification Pipeline
Fit your optimal model on scaled variables.


In [ ]:
from sklearn.ensemble import VotingClassifier

# Scaled variables to prevent leakage
scaler_opt = StandardScaler()
X_tr_opt = scaler_opt.fit_transform(X_tr_w)
X_te_opt = scaler_opt.transform(X_te_w)

# Hyperparameter tuned models
rf_opt = RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_split=2, random_state=42)
svm_opt = SVC(C=5.0, gamma='scale', kernel='rbf', probability=True, random_state=42)
knn_opt = KNeighborsClassifier(n_neighbors=5)

# Soft Voting Ensemble
optimized_ensemble = VotingClassifier(
    estimators=[
        ('rf', rf_opt),
        ('svm', svm_opt),
        ('knn', knn_opt)
    ],
    voting='soft'
)

optimized_ensemble.fit(X_tr_opt, y_tr_w)
y_pred_opt = optimized_ensemble.predict(X_te_opt)
final_accuracy = accuracy_score(y_te_w, y_pred_opt)

print(f"Optimized Ensemble Test Accuracy: {final_accuracy:.4%}")


### Step 5.2: Visualize Confusion Matrix
Plot confusion matrix.


In [ ]:
cm_opt = confusion_matrix(y_te_w, y_pred_opt)
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_w, yticklabels=class_w)
plt.title('Confusion Matrix (Optimized Model)', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()
